# Phase 2 — Synthetic Data + Pretraining (Colab)
**DocLayout-YOLO-Indic**

Runs the Phase 2 data engine and pretraining on Colab. Key design choices for Colab:
- **Code** lives in Drive (`MyDrive/doclayout-yolo-indic`), same as Phase 1.
- **Generation writes to LOCAL `/content`** (fast), then we tar the ~7 GB corpus and copy *one* file to Drive. (Never write 150K small files straight to Drive — it is slow and hits file-count limits.)
- **Data generation** = T4 or any runtime (CPU-bound, parallelised). **Pretraining** = A100.

Runtime: `Runtime → Change runtime type` → **T4** for Cells 0–5, switch to **A100** for Cell 6.

## Cell 0 — Bootstrap (mount Drive, install data-engine deps)

In [1]:
import os, sys, subprocess, json
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))

# Data-engine deps only (CPU). Training deps installed later in Cell 6.
subprocess.run(['pip','install','-q','uharfbuzz','fonttools','pillow','numpy','tqdm'], check=True)

import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
print('vCPUs   :', os.cpu_count())
print('PROJECT_ROOT:', PROJECT_ROOT)

KeyboardInterrupt: 

## Cell 1 — Put the Phase 2 code in Drive (one-time)
Upload `doclayout-yolo-indic-phase2.zip` to your Drive at `MyDrive/` once, then run this to unpack into `PROJECT_ROOT`. (Re-running is safe; it overwrites `src/`.)

If you instead keep the code on GitHub, replace this with a `git clone`/`git pull` into `PROJECT_ROOT`.

In [ ]:
# ── Cell 1: Get the Phase 2 code from GitHub ──
import subprocess, shutil, zipfile
from pathlib import Path

# 👇 set your repo (HTTPS). For a PRIVATE repo, use the token form in the note below.
GITHUB_URL    = "https://github.com/vigneshpalanivelr/mtech-project-aiml.git"
GITHUB_BRANCH = "main"
REPO_LOCAL    = Path("/files")          # fast local clone (re-clone each session is fine)

# clone or pull
if (REPO_LOCAL / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_LOCAL), "pull", "--ff-only"], check=True)
else:
    shutil.rmtree(REPO_LOCAL, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", "-b", GITHUB_BRANCH,
                    GITHUB_URL, str(REPO_LOCAL)], check=True)

# unzip the code bundle
zip_path = REPO_LOCAL / "files" / "doclayout-yolo-indic.zip"
assert zip_path.exists(), f"Zip not found at {zip_path}"
unzip_dir = Path("/content/_unzipped")
shutil.rmtree(unzip_dir, ignore_errors=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(unzip_dir)

# find the folder that contains src/
src_root = next(p.parent.parent for p in unzip_dir.rglob("src/config.py"))
print("src_root:", src_root)  # should now be .../doclayout-yolo-indic  (not .../src)

# sync into PROJECT_ROOT (Drive) so all other cells work unchanged
for item in ["src", "tests", "requirements.txt", "README.md"]:
    s = src_root / item
    d = PROJECT_ROOT / item
    if s.is_dir():
        shutil.rmtree(d, ignore_errors=True)
        shutil.copytree(s, d)
    elif s.exists():
        shutil.copy(s, d)

assert (PROJECT_ROOT / "src" / "synthetic_data" / "generator.py").exists()
print("Code synced to", PROJECT_ROOT, "✓")

## Cell 2 — Fonts (downloads ~5 MB of Noto fonts into Drive, cached)

In [ ]:
%cd {PROJECT_ROOT}
!python -m src.synthetic_data.font_setup

## Cell 3 — Pilot: 200 pages + visual QC (the Week-3 quality gate)
Generates a small batch to **local disk** and shows a contact sheet inline. Eyeball it (ideally with a native reader): shirorekha present, conjuncts/matras correct, Urdu reads right-to-left, boxes tight. Do not proceed to full generation until this looks right.

In [ ]:
%cd {PROJECT_ROOT}
!python -m src.synthetic_data.parallel --num 200 --tag pilot --out /content/pilot --workers $(nproc)

# inline contact sheet from the LOCAL pilot dir
import json
from pathlib import Path
from PIL import Image
from src.synthetic_data.quality_inspector import overlay
root = Path('/content/pilot')
thumbs=[]
for i in range(1, 13):
    rec = json.loads((root/'annotations'/f'pilot_{i:06d}.json').read_text())
    img = overlay(root/'images'/rec['image_file'], rec['annotations'])
    thumbs.append(img.resize((300,300)))
sheet = Image.new('RGB',(4*300,3*300),'white')
for k,t in enumerate(thumbs):
    r,c=divmod(k,4); sheet.paste(t,(c*300,r*300))
sheet.save('/content/pilot_contact_sheet.png')
from IPython.display import Image as IPImage, display
display(IPImage('/content/pilot_contact_sheet.png'))

## Cell 4 — Full corpus → local disk → tar → Drive
~150K pages ≈ **7 GB**, a few hours on an A100 high-RAM runtime (~12 vCPUs). Written to `/content` then packed into a single tar copied to Drive so it survives the session.

Tip: to iterate faster first, generate `--num 30000` (a 30K pilot is enough to confirm pretraining transfers), then scale to 150000.

In [ ]:
%cd {PROJECT_ROOT}
import os
N = 150000   # set to 30000 for a fast first pass
!python -m src.synthetic_data.parallel --num {N} --tag indicsynth --out /content/IndicSynth --workers $(nproc) --chunk 2000

# pack one tar and copy to Drive
import shutil
from pathlib import Path
src_tar = Path('/content/IndicSynth.tar')
!python -c "from src.synthetic_data.parallel import pack_tar; from pathlib import Path; pack_tar(Path('/content/IndicSynth'), Path('/content/IndicSynth.tar'))"
dst = PROJECT_ROOT/'output'/'synthetic'/'IndicSynth.tar'
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(src_tar, dst)
print('Corpus tar in Drive:', dst, f'{dst.stat().st_size/1e9:.1f} GB')

## Cell 4b — (Optional, recommended) Push the corpus to a HuggingFace dataset
Cleaner than a Drive tar for a 7 GB corpus: free, versioned, resumable. Uploads the single tar as one file. Next session you pull it back with `hf_hub_download`. Needs a HF token with **write** access (https://huggingface.co/settings/tokens).

In [ ]:
# --- push (run once, after Cell 4 has produced /content/IndicSynth.tar) ---
!pip install -q huggingface_hub
from huggingface_hub import HfApi, login
from pathlib import Path

HF_TOKEN = ""   # paste a write token, or set os.environ['HF_TOKEN']
HF_REPO  = "VigneshPR/indicsynth-150k"               # change to your namespace
import os; HF_TOKEN = HF_TOKEN or os.environ.get('HF_TOKEN','')
assert HF_TOKEN, "Set a HuggingFace write token first"

login(token=HF_TOKEN)
api = HfApi()
api.create_repo(HF_REPO, repo_type="dataset", private=True, exist_ok=True)
api.upload_file(
    path_or_fileobj="/content/IndicSynth.tar",
    path_in_repo="IndicSynth.tar",
    repo_id=HF_REPO, repo_type="dataset",
)
print("Uploaded -> https://huggingface.co/datasets/" + HF_REPO)

# --- pull (use this at the start of any later session instead of Drive) ---
# from huggingface_hub import hf_hub_download
# tar = hf_hub_download(repo_id=HF_REPO, repo_type="dataset",
#                       filename="IndicSynth.tar", local_dir="/content")
# !tar -xf {tar} -C /content

## Cell 5 — Convert to YOLO format (CPU)
Builds `images/{train,val}`, `labels/{train,val}`, and `data.yaml` on local disk.

In [ ]:
%cd {PROJECT_ROOT}
!python -m src.pretraining.train_synthetic --prepare \
    --coco /content/IndicSynth/indicsynth_coco.json
# the converter reads images from output/synthetic/images by default; point it at the local corpus:
import shutil, json
from pathlib import Path
# (re)run conversion pointing explicitly at the local corpus images
from src.pretraining.train_synthetic import coco_to_yolo
from src.utils.paths import get_output_dir
yaml_path = coco_to_yolo(Path('/content/IndicSynth/indicsynth_coco.json'),
                         Path('/content/IndicSynth/images'),
                         Path('/content/yolo_dataset'))
print('data.yaml ->', yaml_path)
print(yaml_path.read_text())

## Cell 6 — Pretrain on A100  ⚠️ switch runtime to A100 first
Installs training deps + the DocLayout-YOLO package, downloads the **DocSynth300K-pretrained** checkpoint (the correct initializer — see notes), and trains for 30 epochs.

In [ ]:
%cd {PROJECT_ROOT}
!pip install -q torch torchvision ultralytics pycocotools
!pip install -q git+https://github.com/opendatalab/DocLayout-YOLO.git

from src.pretraining.train_synthetic import download_base_checkpoint, train
from pathlib import Path
ckpt = download_base_checkpoint('/content')   # juliozhao/DocLayout-YOLO-DocSynth300K-pretrain
train(Path('/content/yolo_dataset/data.yaml'), ckpt)

## Cell 7 — Back up the pretrained checkpoint to Drive

In [ ]:
import shutil
from pathlib import Path
src = Path('output/checkpoints/doclayout_yolo_indic_pretrained.pt')
if src.exists():
    dst = PROJECT_ROOT/'output'/'checkpoints'/src.name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, dst)
    print('Backed up ->', dst)
else:
    print('No checkpoint found yet — run Cell 6 on an A100 runtime.')

## Datasets for later phases (read now, act when you reach them)

**IndicDLP (Phase 3/4 — fine-tune + eval, *not* needed for Phase 2).**
Do **not** upload your 80 GB `indicdlp.tar` to Drive. It is already at a public URL, so pull it straight into Colab local disk when Phase 3 starts:
```
!wget -q https://objectstore.e2enetworks.net/indic-dlp/indicdlp.tar -O /content/indicdlp.tar
!tar -xf /content/indicdlp.tar -C /content/indicdlp
```
Re-download per session (free, ~10–20 min) rather than parking 80 GB in Drive.

**BaDLAD (Phase 3 — self-training + test).**
Set up Kaggle API access now, grab only the **labeled** split to confirm access; defer the ~4M unlabeled images until Phase 3 and use only a ~200K subset.
```
from google.colab import files; files.upload()   # upload kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
# then download the BaDLAD dataset/competition data into /content
```